<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c13-header.png" alt="Nextia Learning · Document and Multimodal AI" width="100%">

# Extract a table from a public report

**[Extract a table from a public report](https://learning.nextia-ai.com/courses/document-ai/m07/extract-a-table-from-a-public-report/)** · Document and Multimodal AI · Module 7, case study 2 of 2 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** recover a statistics table that runs over two pages of a public report, with the page and the box of every cell, and check it cell by cell against the agency's own CSV files. Then you run it on next month's report.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 1 minute. |
| **Needs** | An internet connection for the setup cells. No account and no key. The first cell installs pdfplumber 0.11.10. |
| **You should know** | [Tables and spreadsheets](https://learning.nextia-ai.com/courses/document-ai/m03/tables-and-spreadsheets/) and [Provenance representation](https://learning.nextia-ai.com/courses/document-ai/m03/provenance-representation/) most of all, and [Business validation](https://learning.nextia-ai.com/courses/document-ai/m05/business-validation/). |
| **Data** | *Crop Production*, USDA National Agricultural Statistics Service, September and October 2026 releases. Public domain. |
| **Tested** | Python 3.12.13 with pdfplumber 0.11.10, in Jupyter on macOS (Apple silicon), from a fresh start |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/document-ai/m07/extract-a-table-from-a-public-report/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C13/public-report-table/public-report-table-solution.ipynb).

## Setup: install the PDF library

Run the next cell. It installs pdfplumber 0.11.10, the PDF library of the course. pdfplumber brings pdfminer.six (the text and its positions) and pypdfium2 (pages as pictures). It needs an internet connection and takes a few seconds.

You should see `pdfplumber 0.11.10`. If you see another version, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
# Setup: install pdfplumber at the course's version.
import importlib.metadata, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pdfplumber==0.11.10"], check=True)
print("pdfplumber", importlib.metadata.version("pdfplumber"))

## Setup: get the reports, the reader and the recordings

Run the next cell. It downloads 7 files (about 3 MB in all) into a new folder `public-report-table`, and checks the SHA-256 checksum of each one:

- `crop0926.pdf` and `crop0926.zip`: the September release of the report, from the agency (USDA NASS). If the agency's server does not answer, the cell takes the same file from the Nextia labs repository; the checksum is the same.
- `crop1026.pdf` and `crop1026.zip`: the October release, the new input at the end.
- `report_table.py`: the reader of this case study (pdfplumber and the standard library only).
- `vision-crop0926.jsonl` and `vision-crop1026.jsonl`: the vision model's answers, recorded. You need no key.

You should see `Ready: 7 files, checksums OK.`

If you see `Could not download`, check your internet connection and run the cell again. In Kaggle, turn on **Internet** in the notebook settings. If you see `wrong checksum`, delete the folder `public-report-table` and run the cell again.

Data: *Crop Production*, USDA National Agricultural Statistics Service (NASS), released September 11 and October 9, 2026. A work of the US Government: public domain. Credit: "USDA National Agricultural Statistics Service".

In [ ]:
# Setup: download each file into public-report-table/ and check it. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

FILES = {
 "crop0926.pdf": [
  "20eae6558f04763ad77976c0538b918beb8e121a35e89697c883f6074ee5868d",
  [
   "https://esmis.nal.usda.gov/sites/default/release-files/796056/crop0926.pdf",
   "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C13/public-report-table/data/crop0926.pdf"
  ]
 ],
 "crop0926.zip": [
  "7e056ccd1e732d0d1c941d948ebdc6691593020f580f85853722026ce1b85a4c",
  [
   "https://esmis.nal.usda.gov/sites/default/release-files/796056/crop0926.zip",
   "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C13/public-report-table/data/crop0926.zip"
  ]
 ],
 "crop1026.pdf": [
  "dc7015e81bd3e75130d445ec4052dcb62813a76c752584054c07bf5b10f67cfc",
  [
   "https://esmis.nal.usda.gov/sites/default/release-files/796101/crop1026.pdf",
   "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C13/public-report-table/data/crop1026.pdf"
  ]
 ],
 "crop1026.zip": [
  "1ccbacbfdd737bdaf7e0af7fac2e08246dd647cd68c456b6d39eb245bedc4f7e",
  [
   "https://esmis.nal.usda.gov/sites/default/release-files/796101/crop1026.zip",
   "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C13/public-report-table/data/crop1026.zip"
  ]
 ],
 "report_table.py": [
  "158cf868a44b5ece3fe1fa2954570b767e923081cf71b0fbd00f153b8dc61516",
  [
   "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C13/public-report-table/report_table.py"
  ]
 ],
 "vision-crop0926.jsonl": [
  "889ec163f0a0d073995e81969549765f248d13c00e8a88ed76f18ad5676026cb",
  [
   "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C13/public-report-table/vision-crop0926.jsonl"
  ]
 ],
 "vision-crop1026.jsonl": [
  "bb45cecf01fb9ba6d034bde119c6093b9ebd9b5fed047806dea35286c6941182",
  [
   "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C13/public-report-table/vision-crop1026.jsonl"
  ]
 ]
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("public-report-table").resolve()
WORK.mkdir(exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def fetch(url, target):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Nextia Learning notebook)"})
    try:
        with urllib.request.urlopen(request, timeout=60) as r:
            target.write_bytes(r.read())
    except OSError:  # Python without certificates (python.org on macOS): try curl
        subprocess.run(["curl", "-fsSL", "-A", "Mozilla/5.0", "-o", str(target), url])

for name, (expected, urls) in FILES.items():
    target = WORK / name
    for source in [HERE / name, HERE / "data" / name, *urls]:
        if target.exists() and sha256(target) == expected:
            break
        if isinstance(source, Path):
            if source.exists():
                shutil.copy(source, target)
        else:
            fetch(source, target)
    if not target.exists():
        raise SystemExit(f"Could not download {name}. Check your internet connection.")
    if sha256(target) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder public-report-table and run again.")
os.chdir(WORK)  # work in this folder: report_table.py is imported from here
print(f"Ready: {len(FILES)} files, checksums OK.")

## Setup: helpers

The next cell loads the reader, `report_table.py`, as `rt`, and opens the September report. It also defines two helpers:

- `show_score(name, results)` prints how many of the agency's cells a reading matches.
- `expect_hash(what, yours, expected)` checks your answer to a task against a hidden fingerprint, without showing the answer.

Run the cell. It prints nothing.

In [ ]:
import importlib, pdfplumber
import report_table as rt
importlib.reload(rt)
# Helpers: check an answer without showing it.
import hashlib, json

def fingerprint(value):
    def norm(v):
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect_hash(what, yours, expected):
    """Compare your result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def show_score(name, results):
    """One line per reading: how many of the agency's cells it matches."""
    s = rt.score(results)
    print(f"{name:34} {s['match']:3} of {s['cells']} cells match"
          f"  (wrong {s['wrong']}, missing {s['missing']}, extra {s['extra']})")

pdf = pdfplumber.open("crop0926.pdf")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Ruth Adeyemi plans storage and trucks for a grain cooperative. Each month the US Department of Agriculture publishes its *Crop Production* report, and her team copies one table into the board's planning sheet: the area, yield and production of every crop. The table runs over two pages. Last spring a number was copied into the wrong column, and nobody saw it for a month.

Ruth asks her analyst, Kofi: "Can a program copy the table, show the board the page and the place of every number, and prove that it copied it right?" The agency also publishes the same table as CSV files. Kofi uses them as the answer sheet: every cell that the program reads must match the agency's CSV. The [lesson page](https://learning.nextia-ai.com/courses/document-ai/m07/extract-a-table-from-a-public-report/) tells the full story; this notebook runs every step.

## 2. Find the table

The report has 47 pages. The table's title is the same each month, but its pages are not. So the reader finds the pages by the title: the first page, and the pages that say "(continued)". Run the cell.

In [ ]:
print(len(pdf.pages), "pages")
pages = rt.find_pages(pdf)
print("table pages:", pages)
for n in pages:
    print(n, pdf.pages[n - 1].extract_text().splitlines()[:2])

> **Check.** You should see `table pages: [25, 26]`, and page 26's title ends with `(continued)`.

Look at the top of page 25. Run the cell: it draws the table's top part as a picture.

In [ ]:
pdf.pages[24].crop((30, 84, 582, 300)).to_image(resolution=90)

Read the picture as Kofi does. The header has two rows: "Area planted" stands over two years. A unit line says `(1,000 acres)`. "Grains and hay" is a bold group row. Alfalfa is indented under "Hay, all". "Corn for grain" has a small mark `1`. And some cells are empty: the cell under 2026 for "Corn for silage" has no value.

## 3. The agency's CSV files

The agency publishes a zip of CSV files with each release: one file per table part, and a help file. Page 26's part is `crop_p26_t196.csv`. Read it as text. Run the cell.

In [ ]:
import zipfile
raw = zipfile.ZipFile("crop0926.zip").read("crop_p26_t196.csv")
try:
    raw.decode("utf-8")
except UnicodeDecodeError as error:
    print("Not UTF-8:", error)
lines = raw.decode("cp1252").splitlines()
for line in lines[3:13]:
    print(line)
print("...")
for line in lines[-6:]:
    print(line)

> **Check.** You should see `Not UTF-8: 'utf-8' codec can't decode byte 0x96 ...`, then the header lines (`"h"`), the units line (`"u"`) and the first data lines (`"d"`).

What this tells us:

- The file is **Windows-1252** text, not UTF-8. Byte 0x96 is the dash in the title. Read as UTF-8, the file stops with an error.
- The second value of each line is its type: `t` title, `h` header, `u` units, `d` data, `f` footnote.
- Footnote marks are written `1/` and `2/`. Indented rows start with two spaces.
- The last data line, `Spearmint oil`, has one value less than the others: the file drops the empty cell at its end.
- The units line has **one value less** than the header. Read by position, `(1,000)` lands under "Yield per acre / 2026", not under the two production columns.

`rt.read_reference()` reads both parts by their title, joins them, and keeps a short line as blanks. Run the cell.

In [ ]:
columns, ref_rows, ref_notes, ref_units = rt.read_reference("crop0926.zip")
print(len(ref_rows), "rows x", len(columns), "columns =", len(ref_rows) * len(columns), "cells")
print(columns)
print("units line, by position:", ref_units)
cbp = {25: columns[:4], 26: columns[5:]}  # the columns of each page, for the simple readers below

> **Check.** You should see `41 rows x 9 columns = 369 cells`. Look at the units: `Yield per acre / 2026` gets `1,000`, and `Production / 2026` gets nothing. The CSV is the answer sheet for the **values**; its units line is not to be trusted by position.

## 4. A baseline: the page as plain text

The quick way: take the page's text, one line per row, and give the numbers to the columns from left to right.

> **Predict.** "Corn for silage" prints `(NA)`, then nothing, then `6,208`, then nothing. Where will the plain-text reading put `6,208`?

Run the cell.

In [ ]:
for line in pdf.pages[24].extract_text().splitlines():
    if line.startswith(("Corn for", "Proso millet")):
        print(line)
text_rows = rt.read_text_lines("crop0926.pdf", cbp)
text_results = rt.compare(text_rows, ref_rows, columns)
show_score("Plain text, split at spaces", text_results)
for r in text_results:
    if r["row"] in ("Corn for silage", "Corn for grain") and r["outcome"] != "match":
        print(f"  {r['row']:16} {r['column']:24} ours {r['ours']!r:10} agency {r['theirs']!r}")

> **Check.** You should see `287 of 369 cells match (wrong 0, missing 65, extra 17)`.

What this tells us. Plain text loses the blank cells: `6,208` moves left into "Area planted / 2026", where the agency has nothing. And "Corn for grain 1" is not "Corn for grain": the footnote mark sticks to the label, so the program cannot find the row at all. No character was misread; the losses come from the structure.

## 5. The PDF library's table finder

pdfplumber has a table finder. With its default settings it looks for lines around every cell. Run the cell.

In [ ]:
grid = pdf.pages[24].extract_table()
print(len(grid), "rows")
print(grid[1])
print(repr(grid[2][0][:60]), "...")
show_score("PDF library, default", rt.compare(rt.read_library("crop0926.pdf", cbp), ref_rows, columns))

> **Check.** You should see `3 rows` and `0 of 369 cells match`. The page has vertical rules but no lines between rows, so the finder makes one tall cell per column. The label column and the last column are missing: no rule stands at the table's outer edges.

Now tell the finder what the page has: columns from the vertical rules, rows from the text lines, and the table's left and right edges. Run the cell.

In [ ]:
settings = {"vertical_strategy": "lines", "horizontal_strategy": "text", "outer_edges": True}
library_results = rt.compare(rt.read_library("crop0926.pdf", cbp, settings), ref_rows, columns)
show_score("PDF library, rules + edges", library_results)
print(sorted({r["row"] for r in library_results if r["outcome"] != "match"}))

> **Check.** You should see `321 of 369 cells match`, and 10 rows that it cannot find: the rows with a footnote mark, such as `Corn for grain` and `Rice`.

The blank cells are right now. Only the footnote marks are left: in the library's text, "Rice ²" is "Rice 2".

## 6. The idea: columns from the rules, rows from the words

The page tells us where its columns are: the thin vertical rules. Each word has a box. So:

1. A word belongs to the column whose rules hold the **middle** of its box. An empty column on a line is a **blank cell**, and it keeps its place.
2. A word printed smaller than 6 points is a **footnote mark**, not part of the label.
3. A bold line with no numbers is a **group**; an indented label is a **part** of the row above it.

Run the cell: it prints the rules and the words of one line, "Corn for silage", with their boxes.

In [ ]:
page = pdf.pages[24]
rules = rt.table_rules(page)
print("column edges:", [round(x) for x in rules["edges"]])
line = [w for w in page.extract_words(extra_attrs=["size"]) if abs(w["top"] - 156.1) < 1]
for w in line:
    middle = (w["x0"] + w["x1"]) / 2
    print(f"{w['text'][:12]:12}  x {w['x0']:6.1f} to {w['x1']:6.1f}  middle {middle:6.1f}  size {w['size']:.0f}")

> **Check.** You should see the edges `[36, 261, 340, 419, 497, 576]`, then `(NA)` with its middle at 326.8 and `6,208` at 482.4.

Do it by hand. 326.8 is between 261 and 340: the first value column, "Area planted / 2025". 482.4 is between 419 and 497: the third, "Area harvested / 2025". Nothing has its middle between 340 and 419, or between 497 and 576: those two cells are blank. This is how `rt.read_table()` reads every line of both pages, and then joins the pages by the row's label. Run it.

In [ ]:
table = rt.read_table("crop0926.pdf")
print("pages", table.pages, "|", len(table.rows), "rows")
print("units from the header:", table.units)
print("footnotes:", table.footnotes)
rice = next(r for r in table.rows if r.label == "Rice")
print(rice.label, "| group", rice.group, "| marks", rice.marks)
for name, cell in rice.cells.items():
    print(f"  {name:22} {cell.text:>8}  {str(cell.unit):16} page {cell.page}  box {cell.box}")
positions_results = rt.compare(table.rows, ref_rows, columns)
show_score("Rules and word positions", positions_results)

> **Check.** You should see `369 of 369 cells match (wrong 0, missing 0, extra 0)`. Rice has the mark `2`, and its yield's unit is `pounds per acre`: the footnote "Yield in pounds" changed it, although the unit column says `cwt`.

Every cell knows its page and its box. Draw Rice's cells on page 26. Run the cell.

In [ ]:
im = pdf.pages[25].crop((30, 84, 582, 240)).to_image(resolution=90)
im.draw_rects([c.box for c in rice.cells.values() if c.page == 26], stroke="green", fill=(0, 160, 0, 40))
im

## 7. Another reader: the vision model

The course's vision model read the two pages as pictures, with a strict JSON schema: every row, its marks, its unit and its values as printed, `""` for a blank cell. Its answers were recorded twice; the notebook replays them. Run the cell.

In [ ]:
import json
records = [json.loads(x) for x in open("vision-crop0926.jsonl")]
for r in records:
    print(f"{r['pass']} page {r['page']}: {r['tokens_in']} + {r['tokens_out']} tokens, {r['seconds']} s, US${r['cost_usd']}")
for run in ("pass1", "pass2"):
    vision_rows = rt.read_vision("vision-crop0926.jsonl", cbp, run=run)
    show_score(f"Vision model ({run}, recorded)", rt.compare(vision_rows, ref_rows, columns))
print("a box for Rice's yield?", next(r for r in vision_rows if r.label == "Rice").cells["Yield per acre / 2025"].box)

> **Check.** You should see `369 of 369` for both passes, about 17 seconds and US$0.02 per page, and `a box for Rice's yield? None`.

The model reads every value right on these clean pages. But it gives values, not places: it cannot show the board where a number is printed. Its answer costs money and time each month, and the two passes could differ on a harder page.

## 8. Checks that need no CSV

Most reports have no CSV. The table can still check itself:

- **Parts add up**: Alfalfa + All other = "Hay, all", for each area and production column.
- **Area × yield = production**, after the units, within the rounding of the printed values.

Run the cell.

In [ ]:
results = rt.checks(table)
print(sum(c["ok"] for c in results), "of", len(results), "checks pass")
for c in results:
    if c["row"] == "Rice":
        print(c)

> **Check.** You should see `79 of 79 checks pass`. For Rice: 2,740 thousand acres × 7,544 pounds per acre ÷ 100 = 206,705.6 thousand cwt, against 206,707 printed.

### A failure case: a digit pair swapped

This is a constructed case. Someone types `1,948,537` instead of `1,984,537` for the wheat production of 2025. Do the checks see it? Run the cell.

In [ ]:
typo = rt.read_table("crop0926.pdf")
wheat = next(r for r in typo.rows if r.label == "Wheat, all")
wheat.cells["Production / 2025"].text = "1,948,537"
for c in rt.checks(typo):
    if not c["ok"]:
        print(c["check"], "|", c["row"], c["column"], "| printed", c["printed"], "computed", c["computed"], "allowed", c["allowed"])

> **Check.** You should see two failed checks: the parts (Winter + Durum + Other spring) give 1,984,537, and area × yield gives about 1,984,945.

Both checks catch it, and they point to the cell. A check cannot say which side is wrong; the box tells a person where to look on the page.

## 9. End to end on new input: the October report

The agency published the next release a month later. Run the whole reading on it: find the table, read it, compare it with October's CSV, run the checks, and list what changed since September. Nothing in the reader changes.

In [ ]:
october = rt.read_table("crop1026.pdf")
oct_columns, oct_ref, _, _ = rt.read_reference("crop1026.zip")
print("pages", october.pages)
show_score("October, rules and word positions", rt.compare(october.rows, oct_ref, oct_columns))
oct_checks = rt.checks(october)
print(sum(c["ok"] for c in oct_checks), "of", len(oct_checks), "checks pass")
changed = rt.changes(table, october)
print(len(changed), "cells changed since September")
for c in changed:
    if c["row"] == "Corn for grain" and c["column"].endswith("2026"):
        print(f"  {c['row']} | {c['column']}: {c['before']} -> {c['after']}  (page {c['page']}, box {c['box']})")

> **Check.** You should see `pages [26, 27]`, `369 of 369 cells match`, `82 of 82 checks pass` and `111 cells changed since September`.

The table moved one page down, and the reader found it by its title. The corn forecast for 2026 went up: 15,799,759 to 16,033,944 thousand bushels. Each change has its page and box, so the board can see it on the new report.

## 10. Your turn

> **Your turn: count the blank cells.** How many cells of September's table are blank (the estimate was not made yet)? Use `table.rows` and `cell.blank`. Put the number in `blank_cells`.

In [ ]:
blank_cells = ...  # the blank cells of September's table

In [ ]:
expect_hash("blank_cells", blank_cells, '9f14025af0065b30')

> **Your turn: change one thing.** The report has the same table in metric units on the next two pages. Its title starts with `"Crop Area Planted and Harvested, Yield, and Production in Metric Units"`. Read it with `rt.read_table(path, title)`, read the reference with `rt.read_reference(zip_path, title)`, and compare. Put the number of matching cells in `metric_match`. Predict first: does the reader need a change?

In [ ]:
METRIC = "Crop Area Planted and Harvested, Yield, and Production in Metric Units"
metric_match = ...  # cells of the metric table that match the agency's CSV

In [ ]:
expect_hash("metric_match", metric_match, '2452984f72ef1195')

> **Your turn: the biggest revision.** Among the cells in `changed` whose column is `"Production / 2026"` and that have a number before and after, which crop changed by the largest share (after ÷ before − 1, ignoring the sign)? Put its label in `biggest_revision`. Use `rt.to_number()` to read a printed number.

In [ ]:
biggest_revision = ...  # the crop label

In [ ]:
expect_hash("biggest_revision", biggest_revision, '5f14428c8909250c')

## 11. Recap

- The table was found by its **title**, not its page number: pages 25–26 in September, 26–27 in October.
- **Plain text** matched 287 of 369 cells: blank cells shift values left, and footnote marks stick to labels. The library's **default** finder matched 0; with the page's rules and edges, 321.
- **Columns from the rules, rows from the words** matched 369 of 369, with a page, a box and a unit for every cell. The **vision model** also matched 369, twice, but with no boxes, for about US$0.04 a release.
- The CSV is a good answer sheet for values, but it has traps of its own: Windows-1252 text, a short last line, and a units line one cell short.
- Without a CSV, the table checks itself: **parts add up** and **area × yield**. A swapped digit pair failed two checks.

Read the [lesson page](https://learning.nextia-ai.com/courses/document-ai/m07/extract-a-table-from-a-public-report/) for the comparison, the recommendation and how to make this a portfolio piece.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/document-ai/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/document-ai/m07/extract-a-table-from-a-public-report/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The report is public: USDA National Agricultural Statistics Service, public domain.